### EMTOOLS -- Postprocessing
This notebook processes semantic segmentations from Uni-EM into instance segmentations. It outputs the measurements defined below as a pandas dataframe, excel spreadsheet, csv, and in various figures.

**Author:** Philip Ruthig, Paul Flechsig Institute, Center of Neuropathology and Brain Research Leipzig

**Contact:** philip.ruthig@medizin.uni-leipzig.de // philip.ruthig@gmail.com

**Publication:**
Please contact me if you want to use this code for any publication.

#### These measures are output for every fiber (inner) and myelin (outer) instance in a paired manner:
- `label`: unique ID for every cell found.
- `area`: area (in pixels) of the given cell in µm²
- `centroid-0` and `centroid-1`: x and y coordinats of the centroid of each cell
- `axis_major_length`: equivalent ellipse major axis length (diameter) in µm
- `axis_minor_length`: equivalent ellipse minor axis length (diameter) in µm
- `eccentricity`: Value between 0 and 1 that defines the non-circularity of a structure. The closer it is to 0, the closer it is to a circle. An eccentricity of 1 means it is a parabola. Eccentricity here is defined as the ratio of the focal distance (distance between focal points) over the major axis length of the ellipse with the same second moment as the binary structure.
- `orientation`: Orientation of the structure in rad, value between -1/2pi and +1/2pi
- `equivalent_diameter_area`: diameter of the circle with an equivalent area to the cell
- `slice`: a slice object to extract this cell from the image

For more detailed info on the extracted values, see the original documentation of scikit-image function used to extract these values:
- https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.regionprops

#### These measures are output for each pair of fiber and myelin:
- `file`: The filename of the image this cell came from
- `gratio`: approximate g ratio of the given cell, here defined as axis_minor_length(fiber)/axis_minor_length/(myelin)
- `orientation_mean_deg`: Mean orientation between the fiber and myelin (in degrees)
- `eccentricity_mean`: Mean eccentricity between the fiber and myelin

In [ ]:
## uncomment for necessary installs

# Python Kernel changed to 3.9.18 (3.9) to allow the module cucim, which has GPU watershed() function
#!pip install tifffile
# !pip install matplotlib
# !pip install pandas
# !pip install scikit-image
# !pip install tqdm
#!pip install colorcet
#!pip install plotly
#!pip install opencv-python
#!pip install cupy-cuda11x
#!pip install cucim
#!pip install numpy

c:\Users\PFI\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [ ]:

import numpy as np
import tifffile as tf
import scipy.ndimage as ndi
import matplotlib.pyplot as plt
import pandas as pd
import math as m
import skimage
import tqdm
import os
import colorcet as cc
import plotly.io as pio
import plotly.express as px
import re
import pickle
import cv2
import time
import cupy as cp
import cupyx.scipy.ndimage as cndi
import dask.array as da

from skimage.measure import regionprops, regionprops_table
from skimage.morphology import disk
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from scipy import ndimage
from cupyx.scipy.ndimage import maximum_filter
from os import listdir
from os.path import isfile, join
%matplotlib inline

In [ ]:
def resolve_undersegmentation(outer_labels_gpu,inner_labels_gpu):
    '''
    This function resolves undersegmented 'kissing' cells. This function assumes that inner_labels never touch, but outer_labels do.
    Each outer area of a cell must touch the corresponding inner area.

    outer_labels = outer labels with undersegmented kissing cells that need to be seperated
    inner_labels = inner labels without undersegmented kissing cells

    returns: Two arrays of the same shape, with uniform labels across both images. 
    '''

    outer_labels = cp.asnumpy(outer_labels_gpu)
    inner_labels = cp.asnumpy(inner_labels_gpu)
    # do distance transformation of combined binary image of outer+inner
    seg_dist = ndi.distance_transform_edt(outer_labels+inner_labels)
    # Generate the markers as local maxima of the distance to the background
    coords = peak_local_max(seg_dist, footprint=np.ones((3, 3)), labels=ndi.label(inner_labels)[0], num_peaks_per_label=1)
    # initializie empty mask 
    mask = np.zeros(seg_dist.shape, dtype=bool)
    # insert maximum points into the empty array
    mask[tuple(coords.T)] = True
    # label each maximum
    markers, n = ndi.label(mask)
    # perform watershed on outer_labels and inner_labels combined
    outer_cells = watershed(-seg_dist, markers, mask=(outer_labels+inner_labels))
    inner_cells = np.copy(outer_cells)
    # sort the cells (which are now labeled with the same label inner and outer) back into inner and outer labels
    inner_cells[inner_labels==0] = 0
    outer_cells[inner_labels==True] = 0
    return cp.asarray(outer_cells),cp.asarray(inner_cells)

# GPU Version of resolve_undersegmentation()
def resolve_undersegmentation_gpu(outer_labels_gpu, inner_labels_gpu):
    '''
    Resolves undersegmented 'kissing' cells using watershed and peak_local_max.
    This function assumes that inner_labels never touch, but outer_labels do.
    '''
    # Step 1: Distance transformation of the combined binary image of outer + inner (GPU-based)
    sum_labels_gpu = outer_labels_gpu + inner_labels_gpu
    seg_dist_gpu = cndi.distance_transform_edt(sum_labels_gpu)
    inner_labeled_gpu, _ = cndi.label(inner_labels_gpu)

    # Step 2: Convert to NumPy for peak_local_max, since it requires a CPU-based array
    seg_dist_cpu = cp.asnumpy(seg_dist_gpu)  # Convert to NumPy array for skimage
    inner_labeled_cpu = cp.asnumpy(inner_labeled_gpu)

    # Step 3: Generate markers as local maxima on CPU, then convert back to GPU
    coords_cpu = peak_local_max(seg_dist_cpu, footprint=np.ones((3, 3)), labels=inner_labeled_cpu, num_peaks_per_label=1)

    # Convert coordinates back to GPU
    coords_gpu = cp.asarray(coords_cpu)

    # Step 4: Create an empty mask on GPU for the markers
    mask_gpu = cp.zeros(seg_dist_gpu.shape, dtype=cp.bool_)

    # Insert maximum points into the mask (GPU-based)
    if coords_gpu.size > 0:
        mask_gpu[coords_gpu[:, 0], coords_gpu[:, 1]] = True

    # Step 5: Label the maxima (markers) on the mask (GPU-based)
    markers_gpu, _ = cndi.label(mask_gpu)

    # Step 6: Perform watershed segmentation on the combined image (outer + inner) on CPU
    seg_dist = cp.asnumpy(seg_dist_gpu)  # Convert the distance transform back to CPU
    markers = cp.asnumpy(markers_gpu)   # Convert the markers to CPU for watershed
    sum_labels = cp.asnumpy(sum_labels_gpu)  # Convert sum_labels to CPU

    # Perform watershed on CPU (since we want it to run on CPU)
    outer_cells = watershed(-seg_dist, markers, mask=sum_labels)
    outer_cells_gpu = cp.asarray(outer_cells)

    # Step 7: Copy the watershed result for the inner labels
    inner_cells_gpu = cp.copy(outer_cells_gpu)

    # Step 8: Apply masks (only retain outer cells and inner cells correctly separated)
    inner_mask_gpu = cp.equal(inner_labels_gpu, 0)
    outer_mask_gpu = cp.equal(inner_labels_gpu, True)

    inner_cells_gpu[inner_mask_gpu] = 0
    outer_cells_gpu[outer_mask_gpu] = 0

    # Return the results on GPU
    return outer_cells_gpu, inner_cells_gpu
 
def keep_largest_structure(boolean_array):

    # Label each connected component in the boolean array
    labeled_array, num_features = ndimage.label(boolean_array)

    # Calculate the size of each labeled component
    component_sizes = np.bincount(labeled_array.ravel())

    # Find the index of the largest component
    largest_component_index = np.argmax(component_sizes[1:]) + 1

    # Create a boolean mask to keep only the largest component
    largest_component_mask = labeled_array == largest_component_index

    # Apply the mask to the boolean array
    boolean_array[largest_component_mask] = True
    boolean_array[~largest_component_mask] = False

    return boolean_array


def keep_largest_structure_gpu(boolean_array_gpu, labeled_array_gpu):
    component_sizes_gpu = cp.bincount(labeled_array_gpu.ravel())
    largest_component_index_gpu = cp.argmax(component_sizes_gpu[1:]) + 1
    largest_component_mask_gpu = labeled_array_gpu == largest_component_index_gpu
    return largest_component_mask_gpu.astype(cp.uint16)

def remove_small_objects_gpu(inner_labeled, outer_labeled, min_size=5):
    ids = cp.unique(inner_labeled)

    max_id = ids.max()
    counts = cp.bincount(inner_labeled.ravel(), minlength=int(max_id)+1)

    small_ids = ids[(counts[ids] < min_size) & (ids != 0)]
    small_cnt = counts[(counts[ids] < min_size) & (ids != 0)]

    mask_inner = cp.isin(inner_labeled, small_ids)
    mask_outer = cp.isin(outer_labeled, small_ids)

    inner_labeled[mask_inner] = 0
    outer_labeled[mask_outer] = 0

    return (
        inner_labeled,
        outer_labeled,
        cp.asnumpy(small_ids).tolist()
    )

def find_numbers(string):
    """
    Finds up to 4-digit numbers in a string.
    Parameters:
        string
    
        Returns:
        list of numbers
    """
    pattern = r'\d{1,4}'  # Regular expression pattern for matching numbers with 1 to 4 digits
    numbers = re.findall(pattern, string)
    numbers = [int(num) for num in numbers]  # Convert numbers to integers
    return numbers

def z_score_dataframe(dataframe):
    """
    Z-scores a Pandas DataFrame.

    Parameters:
        dataframe (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: The z-scored DataFrame.
    """
    z_scored_dataframe = (dataframe - dataframe.mean()) / dataframe.std(ddof=0)
    return z_scored_dataframe

def remove_whitespaces(string):
    return "".join(string.split())

def reconstruct_images(folder,img_coord_list,img_original_shape_list,img_original_name_list,save_dir,batch_size):
    i=0
    for orig_img_name in np.unique(np.array(img_original_name_list)):
        rec_img = np.zeros(img_original_shape_list[i], dtype='uint8')
        print(f"resaving image: {orig_img_name}")
        rec_img_padded = np.pad(rec_img,batch_size,mode='reflect')
        
        for img_coords in tqdm.tqdm(img_coord_list):
            temp_subimg = cv2.imread(f"{folder}\\{orig_img_name}{remove_whitespaces(str(img_coords))}.png")
            if type(temp_subimg) == type(None):
                continue
            y_start, y_end, x_start, x_end = img_coords
            rec_img_padded[y_start:y_end,x_start:x_end] = temp_subimg[:,:,0]

        ## crop back to region without pads
        rec_img_cropped = rec_img_padded[batch_size:-batch_size,batch_size:-batch_size]
        tf.imwrite(save_dir+orig_img_name+".tif",rec_img_cropped.astype('uint8'))
        print('writing fused image to: ' + str(save_dir+orig_img_name+".tif"))
        i+=1

def brute_force_binary_dilation(input_gpu, iterations):
    dilated_gpu = cp.copy(input_gpu)
    for _ in range(iterations):
        dilated_gpu = cndi.binary_dilation(dilated_gpu)
    return dilated_gpu

def process_cells_on_gpu(inner_labeled_gpu, outer_labeled_gpu, myelin_thresh=myelin_thresh):
    #get unique IDs
    unique_ids_gpu = cp.unique(inner_labeled_gpu)
    print(f"Number of axon labels to process: {len(unique_ids_gpu)}")
    
    # iterate through all IDs
    for id_gpu in tqdm.tqdm(unique_ids_gpu):
        current_cell_gpu = inner_labeled_gpu == id_gpu
        # dilate and remove original cell
        current_cell_dil_gpu = brute_force_binary_dilation(current_cell_gpu, iterations=12)
        current_cell_dil_only_gpu = current_cell_dil_gpu.astype('uint8') - current_cell_gpu.astype('uint8')
        # check overlap
        dil_overlap_gpu = cp.logical_and(current_cell_dil_only_gpu, outer_labeled_gpu == id_gpu)
        perc_overlap_gpu = cp.count_nonzero(dil_overlap_gpu) / cp.count_nonzero(current_cell_dil_only_gpu)
        # If overlap below threshold, remove cell
        if perc_overlap_gpu < myelin_thresh:
            inner_labeled_gpu[inner_labeled_gpu == id_gpu] = 0
            outer_labeled_gpu[outer_labeled_gpu == id_gpu] = 0
    return inner_labeled_gpu, outer_labeled_gpu 

def binary_fill_holes_gpu(mask_gpu):
    inverted = ~mask_gpu
    border = cp.zeros_like(mask_gpu, dtype=bool)
    border[0, :] = True
    border[-1, :] = True
    border[:, 0] = True
    border[:, -1] = True

    markers = cp.logical_and(border, inverted)
    labeled, _ = cndi.label(markers)
    outside = cndi.binary_propagation(labeled > 0, mask=inverted)
    filled = mask_gpu | ~outside
    return filled

def mk_dir(directory_path):
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)

# define qualitative colormap
glasbey = cc.cm.glasbey_dark_r
glasbey.set_under(color="black")

In [ ]:
path_preprocessed_images = r"1_preprocessed\\"
path_raw_predictions = r"2_predicted\\"
path_reconstructed_predictions = r"2_predicted_reconstructed\\"
path_results = r"3_postprocessed\\"

In [4]:
### USER INPUTS
remove_cells_without_myelin = True # if True, removes a lot of false positives. If False, keeps unmyelinated cells.
neuropatho = True # take threshold values that are better for mouse samples
px_size = 4.3*4 # pixel size in nm, multiplied by the downscaling factor applied in the preprocessing step
threshold_myelin = 50
threshold_fiber_upper = 40 
threshold_fiber_lower = 28
overlap = 200 # overlap defined in the preprocessing step 1

myelin_thresh = 0.3 # which percentage does each axon need to be covered by myelin

if neuropatho == True: # optimized to have unmyelinated cells in there, and less degraded cells.
    threshold_fiber_upper = 29 
    threshold_fiber_lower = 8

## for troubleshooting purposes only
crop = False # for quick troubleshooting with huge images
troubleshoot_small_big_gratios = False # Only set to True if you get weird (lots of very small (<0.1) and/or big (>0.9)) gratios
plot_all = False # if True, plots a lot more intermediate steps

In [ ]:
### reconstruct images from small image slices and re-save them.
# load metadata from pkl files
with open(r"img_name_list","rb") as fp:
    img_name_list = pickle.load(fp)

with open(r"img_coord_list","rb") as fp:
    img_coord_list = pickle.load(fp)

with open(r"img_original_shape_list","rb") as fp:
    img_original_shape_list = pickle.load(fp)

with open(r"img_original_name_list","rb") as fp:
    img_original_name_list = pickle.load(fp)

with open(r"batch_size","rb") as fp:
    batch_size = pickle.load(fp)

# reconstruct preprocessed and predicted images to full size
reconstruct_images(folder="1_preprocessed",
                  img_coord_list=img_coord_list,
                  img_original_shape_list=img_original_shape_list,
                  img_original_name_list=img_original_name_list,
                  save_dir=r"1_preprocessed_reconstructed\\",
                  batch_size=batch_size)

reconstruct_images(folder="2_predicted",
                  img_coord_list=img_coord_list,
                  img_original_shape_list=img_original_shape_list,
                  img_original_name_list=img_original_name_list,
                  save_dir=r"2_predicted_reconstructed\\",
                  batch_size=batch_size)

In [ ]:
# Tilo Reinert (April 2025 for PLOS Biology Revision):
# Code additions to speed up runtime using GPU for many tasks => named with extension _gpu
# The script was executed on the HP Z6 workstation "Z6@3DREM"  (SH5OAL7) with NVIDIA RTX A5000 GPU
# Python: 3.11.7
# CUDA Version: 12.4.99
# Cuda compilation tools, release 12.4, V12.4.99
# Build cuda_12.4.r12.4/compiler.33961263_0

i = 0 # index, number of pictures already run
prediction_files = [
    f for f in listdir(path_reconstructed_predictions) 
    if isfile(join(path_reconstructed_predictions, f)) 
    and f.lower().endswith(('.tif', '.tiff')) 
    and f != ".gitkeep"
]
file_cnt = 0
for file in prediction_files:
    # Fortschritt ausgeben
    print(file, "Size in px: ", img.shape)
    file_cnt +=1
    progress = int(100 * file_cnt / len(prediction_files))
    print(f">>>>>>>>>>> Progress: {progress}% ")
    img = tf.imread(path_reconstructed_predictions + file)
    

    if crop == True:
        img = img[000:2000,000:2000]
    
    if plot_all == True:
        # plot raw image
        fig, ax = plt.subplots(ncols=1,figsize=(8,8))
        ax.imshow(img, cmap='viridis')
        ax.set_title('Uni-EM Presegmentation')
        plt.show()
        plt.hist(img.ravel(), bins=256, range=(0, 255),)
        plt.show()

    # split different labels
    outer_gpu = cp.zeros_like(img)
    inner_gpu = cp.zeros_like(img)
    
    if neuropatho == False:
        outer_gpu[img>threshold_myelin] = 1
    if neuropatho == True:
        outer_gpu[img>threshold_fiber_upper] = 1

    inner_gpu[img>threshold_fiber_lower] = 1
    inner_gpu[img<threshold_fiber_lower] = 0
    inner_gpu[outer_gpu==True]=0

    if plot_all == True:
        fig, axs = plt.subplots(ncols=2,figsize=(12,12))
        axs[0].imshow(outer_gpu.get(), cmap='gray')
        axs[1].imshow(inner_gpu.get(), cmap='gray')
        axs[0].set_title('Outer channel of pre-segmentation')
        axs[1].set_title('Inner channel of pre-segmentation')
        plt.show()

    # binary opening to get rid of small speckles
    struk = disk(2)
    struk_gpu = cp.asarray(struk)
    inner_gpu = cndi.binary_opening(inner_gpu,structure=struk_gpu)

    print(f"start filling holes")
    inner_gpu = binary_fill_holes_gpu(inner_gpu)

    # dilate inner, then restrict it to everywhere where outer isnt true. 
    # This is to make sure they are in contact and can be seperated by watershed later on
    struk = disk(3)
    struk_gpu = cp.asarray(struk)
    inner_gpu = cndi.binary_dilation(inner_gpu,structure=struk_gpu)
 
    if plot_all == True:
        # plot overlay of inner + outer as sanity check
        plt.title('after dilation of inner & restriction to outer channel')
        plt.imshow(np.ma.array(inner_gpu.get(),mask=inner_gpu.get()==0),interpolation='None',cmap='tab20')
        plt.imshow(np.ma.array(outer_gpu.get(),mask=outer_gpu.get()==0),interpolation='None',cmap='gray')
        plt.show()

    #### Cleanup of segmented data
    #### Find corresponding outer cells for every inner cell
    print(f"start resolve_undersegmentation()")
    
    outer_labeled_gpu, inner_labeled_gpu = resolve_undersegmentation_gpu(outer_gpu.astype("bool_"),inner_gpu.astype("bool_"))

    both_labeled_gpu = outer_labeled_gpu + inner_labeled_gpu
    if plot_all == True:
        print('after relabeling first time')
        plt.axis("off")
        plt.imshow(both_labeled_gpu.get(),cmap='viridis')
        plt.show()
    
    # remove cells that intersect with the border of the image
    border_mask_gpu = cp.zeros_like(both_labeled_gpu, dtype=cp.bool_)
    border_mask = cp.asnumpy(border_mask_gpu)
    border_mask = ndi.binary_dilation(border_mask, iterations=5, border_value=True)
    border_mask_gpu = cp.asarray(border_mask)

    ids = cp.unique(both_labeled_gpu)
    print(f"start removing border cells")
    print(f"Number of labels: {len(ids)}")

    # Mask border regions
    border_ids = cp.unique(both_labeled_gpu[border_mask_gpu])

    # remove background id (zero)
    border_ids = border_ids[border_ids != 0]

    # Set border ids to 0 in both_labeled_gpu
    mask = cp.isin(both_labeled_gpu, border_ids)
    both_labeled_gpu[mask] = 0
  
    outer_labeled_gpu = cp.copy(both_labeled_gpu)
    inner_labeled_gpu = cp.copy(both_labeled_gpu)
    
    # seperate inner and outer back out
    outer_labeled_gpu[inner_gpu == True] = 0
    inner_labeled_gpu[inner_gpu == 0] = 0
 
    if plot_all == True:
        inner_labeled = cp.asnumpy(inner_labeled_gpu)
        plt.axis("off")
        plt.imshow(inner_labeled.astype('bool'),cmap='gray')
        plt.show()

    # at this point, some of the cells do not have myelin. Remove them if needed.
    
    if remove_cells_without_myelin == True:
        print(f"start removing axons w/o myelin")
        idx_list = []

        # Tilo faster version remove axon w/o myelin
        inner_ids = cp.unique(inner_labeled_gpu)
        outer_ids = cp.unique(outer_labeled_gpu)

        remove_ids = cp.setdiff1d(inner_ids, outer_ids)
        mask = cp.isin(inner_labeled_gpu, remove_ids)
        inner_labeled_gpu[mask] = 0

        idx_list = cp.asnumpy(remove_ids).tolist()

    #re-label so the labels are uniform again.
    print(f"start resolve undersegmentation again")
    outer_labeled_gpu,inner_labeled_gpu = resolve_undersegmentation_gpu(outer_labeled_gpu.astype('bool_'),inner_labeled_gpu.astype('bool_'))

    #### Post-Process filtering
    # START optimized code using GPU:     keep only the biggest one - Inner
    def process_inout_labels_gpu(inout_labeled_gpu):         # This function is an optimized code for calculating inner_labeled and outer_labeled
        unique_ids = cp.unique(inout_labeled_gpu).get()

        # Allocate result array
        result_gpu = cp.zeros_like(inout_labeled_gpu, dtype=cp.uint16)
        for id in tqdm.tqdm(unique_ids):
            if id == 0:
                continue  # Skip background

            # Create mask for current label
            current_id_mask_gpu = inout_labeled_gpu == id

            # Label subcomponents
            current_id_mask_labeled_gpu, n = cndi.label(current_id_mask_gpu)

            if n > 1:
                # Keep only the largest connected part
                largest_component_gpu = keep_largest_structure_gpu(current_id_mask_gpu, current_id_mask_labeled_gpu)

                # Multiply mask by original id
                largest_component_gpu *= id

                # Insert into result
                result_gpu[current_id_mask_gpu] = 0
                result_gpu += largest_component_gpu
            else:
                result_gpu[current_id_mask_gpu] = id

        return result_gpu

    print(f"start keeping biggest - Inner (GPU Version)")
    inner_labeled_gpu = process_inout_labels_gpu(inner_labeled_gpu)
    print(f"start keeping biggest - Outer (GPU Version)")
    outer_labeled_gpu = process_inout_labels_gpu(outer_labeled_gpu)
    # END optimized code using GPU:      keep only the biggest one - Outer

    print(f"start remove unreasonable small fibers")

    inner_labeled_gpu, outer_labeled_gpu, deleted_ids = remove_small_objects_gpu(inner_labeled_gpu, outer_labeled_gpu)

    #<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

    print(f"start remove if inner is not enclosed")
    inner_labeled_gpu, outer_labeled_gpu = process_cells_on_gpu(inner_labeled_gpu, outer_labeled_gpu)

    inner_labeled = cp.asnumpy(inner_labeled_gpu)
    outer_labeled = cp.asnumpy(outer_labeled_gpu)
    table = regionprops_table(label_image=inner_labeled,
                                properties=('label',
                                            'area',
                                            'centroid',
                                            'axis_major_length',
                                            'axis_minor_length',
                                            'eccentricity',
                                            'orientation',
                                            'slice',
                                            ))
                            

    measurements_inner = pd.DataFrame(table)


    if plot_all == True:
        # plot centroids, measurements, major and minor axis on image to sanity check
        regions = regionprops(inner_labeled)

        fig, ax = plt.subplots(figsize=(8,8))
        ax.imshow(inner_labeled.astype('bool'), cmap='gray')

        for props in regions:
            y0, x0 = props.centroid
            orientation = props.orientation
            x1 = x0 + m.cos(orientation) * 0.5 * props.axis_minor_length
            y1 = y0 - m.sin(orientation) * 0.5 * props.axis_minor_length
            x2 = x0 - m.sin(orientation) * 0.5 * props.axis_major_length
            y2 = y0 - m.cos(orientation) * 0.5 * props.axis_major_length

            ax.plot((x0, x1), (y0, y1), '-r', linewidth=2.5)
            ax.plot((x0, x2), (y0, y2), '-r', linewidth=2.5)
            ax.plot(x0, y0, '.g', markersize=10)

            minr, minc, maxr, maxc = props.bbox
            bx = (minc, maxc, maxc, minc, minc)
            by = (minr, minr, maxr, maxr, minr)
            ax.plot(bx, by, '-b', linewidth=1.5)

        ax.set_axis_off()
        ax.axis((0, 1000, 1000, 0))
        plt.show()

    table = regionprops_table(label_image=outer_labeled,
                                # intensity_image=img_intensity,
                                properties=('label',
                                            'area',
                                            'centroid',
                                            'axis_major_length',
                                            'axis_minor_length',
                                            'eccentricity',
                                            'orientation',
                                            'slice',
                                            ))
                        

    measurements_outer = pd.DataFrame(table)

    if plot_all == True:
        # now do the same for every outer cell:
        # plot centroids, measurements, major and minor axis on image to sanity check
        regions = regionprops(outer_labeled)

        fig, ax = plt.subplots(figsize=(8,8))
        ax.imshow(outer_labeled.astype('bool'), cmap='gray')

        for props in regions:
            y0, x0 = props.centroid
            orientation = props.orientation
            x1 = x0 + m.cos(orientation) * 0.5 * props.axis_minor_length
            y1 = y0 - m.sin(orientation) * 0.5 * props.axis_minor_length
            x2 = x0 - m.sin(orientation) * 0.5 * props.axis_major_length
            y2 = y0 - m.cos(orientation) * 0.5 * props.axis_major_length

            ax.plot((x0, x1), (y0, y1), '-r', linewidth=2.5)
            ax.plot((x0, x2), (y0, y2), '-r', linewidth=2.5)
            ax.plot(x0, y0, '.g', markersize=10)

            minr, minc, maxr, maxc = props.bbox
            bx = (minc, maxc, maxc, minc, minc)
            by = (minr, minr, maxr, maxr, minr)
            ax.plot(bx, by, '-b', linewidth=1.5)

        ax.set_axis_off()
        ax.axis((0, 1000, 1000, 0))
        plt.show()

    #re-label columns so they are accurate
    measurements_outer.columns = ['outer_' + col for col in measurements_outer.columns]
    measurements_inner.columns = ['inner_' + col for col in measurements_inner.columns]

    #put them in a single dataframe
    measurements = pd.concat([measurements_inner,measurements_outer],axis=1)
    measurements['file'] = str(file)
    # measurements['is_myelinated'] = measurements['outer_intensity_mean'] < 105

    # correct all lengths and areas to the right size
    factor_length = px_size / 1000 # 10^3 is from nm to µm
    factor_area = px_size * px_size / 1000000 # 10^6 is from nm² to µm²
    measurements['inner_area'] *= factor_area
    measurements['outer_area'] *= factor_area
    measurements['inner_axis_minor_length'] *= factor_length
    measurements['outer_axis_minor_length'] *= factor_length
    measurements['inner_axis_major_length'] *= factor_length
    measurements['outer_axis_major_length'] *= factor_length

    #add gratio to combined dataframe
    measurements['gratio'] = measurements['inner_axis_minor_length']/measurements['outer_axis_minor_length']
    
    # remove all cells with gratio >1 (since that is impossible)
    index_gratio_greater_than_1 = list(measurements[measurements['gratio'] > 1].index)
    index_gratio_greater_than_1 = [index + 1 for index in index_gratio_greater_than_1]
    print(f"Removing cells because gratio >1: {index_gratio_greater_than_1}")
    measurements = measurements[measurements['gratio'] <= 1]

    # remove them from the images as well
    for id in np.unique(index_gratio_greater_than_1):
        inner_labeled[inner_labeled==id]=0
        outer_labeled[outer_labeled==id]=0
    
    # plot final segmentation
    # generate 2px outline of inner and outer area
    outer_eroded = ndi.binary_erosion(outer_labeled,structure=disk(3))
    ero = np.logical_xor(outer_labeled.astype('bool'),outer_eroded)
    vmin = 0
    vmax = inner_labeled.max()
    plt.figure(figsize=(12,12))
    plt.title('Final Segmentation')
    plt.imshow(np.ma.array(inner_labeled + outer_labeled, mask=(inner_labeled + outer_labeled) == 0), cmap=glasbey,vmin=vmin,vmax=vmax)
    plt.imshow(np.ma.array(ero, mask=(ero) == 0), interpolation='None', cmap='gray',)
    plt.savefig(path_results + file + "_labeled_cells_gpu.png",dpi=600)
    plt.close()

    #add orientation and eccentricity means between fiber and myelin. Also convert orientation to degrees from rad.
    measurements['orientation_mean_deg'] = np.degrees((measurements['inner_orientation']+measurements['outer_orientation'])/2)
    measurements['eccentricity_mean'] = ((measurements['inner_eccentricity']+measurements['outer_eccentricity'])/2)

    measurements.replace([np.inf, -np.inf], np.nan, inplace=True)
    if i == 0:
        measurements_all = measurements.copy()
    
    measurements_all = pd.concat([measurements_all,measurements],ignore_index=True)
    
    # Plot the histogram of g ratios
    plt.hist(measurements['gratio'],bins=50, edgecolor='black', color='darkblue', alpha=0.6)

    # Customize plot elements
    plt.ylabel('Number of cells')
    plt.xlabel('G Ratio')
    plt.grid(True, linestyle='--', alpha=0.8, which='both')

    # Add additional plot elements
    plt.axvline(measurements['gratio'].mean(), color='red', linestyle='--', label='Mean')
    plt.axvline(measurements['gratio'].median(), color='darkred', linestyle='--', label='Median')
    plt.legend()

    # Show the plot
    plt.tight_layout()
    plt.savefig(path_results + file + "_g_ratio_gpu.png",dpi=500)
    plt.close()

    tf.imwrite(path_results + file + "_inner_labeled_gpu.tif", inner_labeled)
    tf.imwrite(path_results + file + "_outer_labeled_gpu.tif", outer_labeled)

    if troubleshoot_small_big_gratios == True:
        ### troubleshoot 0/1 g ratio artifact
        measurements_smallgratio_slices = measurements[measurements['gratio'] < 0.3]
        measurements_biggratio_slices = measurements[measurements['gratio'] > 0.9]
        slice_list_big = measurements_biggratio_slices['inner_slice'].tolist()
        slice_list_small = measurements_smallgratio_slices['inner_slice'].tolist()
        p = 5 # p for padding
        for yx in slice_list_small:
            print('plotting small g ratios')
            print(str(yx))
            yx = find_numbers(str(yx)) # converts slice object to a list of numbers
            outer_eroded = ndi.binary_erosion(outer_labeled[yx[0]-p:yx[1]+p,yx[2]-p:yx[3]+p],structure=disk(1))
            ero = np.logical_xor(outer_labeled[yx[0]-p:yx[1]+p,yx[2]-p:yx[3]+p].astype('bool'),outer_eroded)
            plt.imshow(np.ma.array(inner_labeled[yx[0]-p:yx[1]+p,yx[2]-p:yx[3]+p] + outer_labeled[yx[0]-p:yx[1]+p,yx[2]-p:yx[3]+p],
                                   mask=(inner_labeled[yx[0]-p:yx[1]+p,yx[2]-p:yx[3]+p] + outer_labeled[yx[0]-p:yx[1]+p,yx[2]-p:yx[3]+p]) == 0),
                                   cmap=glasbey,
                                   vmin=vmin,vmax=vmax)
            plt.imshow(np.ma.array(ero, mask=(ero) == 0), interpolation='None', cmap='gray',)
            plt.axis('off')
            plt.savefig(path_results + "_gratio_small" + str(yx) + file + ".png")
            plt.show()

        for yx in slice_list_big:
            print('plotting large g ratios')
            print(str(yx))
            yx = find_numbers(str(yx)) # converts slice object to a list of numbers
            outer_eroded = ndi.binary_erosion(outer_labeled[yx[0]-p:yx[1]+p,yx[2]-p:yx[3]+p],structure=disk(1))
            ero = np.logical_xor(outer_labeled[yx[0]-p:yx[1]+p,yx[2]-p:yx[3]+p].astype('bool'),outer_eroded)
            plt.imshow(np.ma.array(inner_labeled[yx[0]-p:yx[1]+p,yx[2]-p:yx[3]+p] + outer_labeled[yx[0]-p:yx[1]+p,yx[2]-p:yx[3]+p],
                                   mask=(inner_labeled[yx[0]-p:yx[1]+p,yx[2]-p:yx[3]+p] + outer_labeled[yx[0]-p:yx[1]+p,yx[2]-p:yx[3]+p]) == 0),
                                   cmap=glasbey,
                                   vmin=vmin,vmax=vmax)
            plt.imshow(np.ma.array(ero, mask=(ero) == 0), interpolation='None', cmap='gray',)
            plt.axis('off')
            plt.savefig(path_results + "_gratio_big" + str(yx) + file + ".png")
            plt.show()
    i +=1
    if crop == True:
        break

### Save Results to Disk

In [ ]:
measurements_all.to_csv(path_results + "all_results.csv")
measurements_all.to_excel(path_results + "all_results.xlsx")

In [ ]:
# Re-Open the results
measurements_all = pd.read_csv(path_results + "all_results.csv")


In [ ]:
# Plot the histogram of g inner inensity means
plt.hist(measurements_all['inner_intensity_mean'],bins=50, edgecolor='black', color='darkblue', alpha=0.6)

# Customize plot elements
plt.ylabel('Number of cells')
plt.xlabel('Intensity (inner)')
plt.grid(True, linestyle='--', alpha=0.4, which='both')
# plt.xticks(np.arange(0, 1, 5))
# plt.yticks(np.arange(0, 21, 2))

# Add additional plot elements
plt.axvline(measurements_all['inner_intensity_mean'].mean(), color='red', linestyle='--', label='Mean')
plt.axvline(measurements_all['inner_intensity_mean'].median(), color='darkred', linestyle='--', label='Median')
plt.legend()

# Show the plot
plt.tight_layout()
plt.savefig(path_results + "_inner_intensities.png",dpi=500)
plt.show()

In [ ]:
# Plot the histogram of outer intensiy means
plt.hist(measurements_all['outer_intensity_mean'],bins=50, edgecolor='black', color='darkblue', alpha=0.6)

# Customize plot elements
plt.ylabel('Number of cells')
plt.xlabel('Intensity (outer)')
plt.grid(True, linestyle='--', alpha=0.4, which='both')

# Add additional plot elements
plt.axvline(measurements_all['outer_intensity_mean'].mean(), color='red', linestyle='--', label='Mean')
plt.axvline(measurements_all['outer_intensity_mean'].median(), color='darkred', linestyle='--', label='Median')
plt.legend()
# Show the plot
plt.tight_layout()
plt.savefig(path_results + "outer_intensities.png",dpi=500)
plt.show()

In [ ]:
# Plot the histogram of g ratios
plt.hist(measurements_all['gratio'],bins=50, edgecolor='black', color='darkblue', alpha=0.6)

# Customize plot elements
plt.ylabel('Number of cells')
plt.xlabel('G Ratio')
plt.grid(True, linestyle='--', alpha=0.4, which='both')

# Add additional plot elements
plt.axvline(measurements_all['gratio'].mean(), color='red', linestyle='--', label='Mean')
plt.axvline(measurements_all['gratio'].median(), color='darkred', linestyle='--', label='Median')
plt.legend()

# Show the plot
plt.tight_layout()
plt.savefig(path_results + "_all_g_ratio.png",dpi=500)
plt.show()

In [ ]:
# Plot the histogram of eccentricity
plt.hist(measurements_all['inner_eccentricity'],bins=50, edgecolor='black', color='darkblue', alpha=0.6)

# Customize plot elements
plt.ylabel('Number of cells')
plt.xlabel('Inner eccentricity')
plt.grid(True, linestyle='--', alpha=0.4, which='both')

# Add additional plot elements
plt.axvline(measurements_all['inner_eccentricity'].mean(), color='red', linestyle='--', label='Mean')
plt.axvline(measurements_all['inner_eccentricity'].median(), color='darkred', linestyle='--', label='Median')
plt.legend()

# Show the plot
plt.tight_layout()
plt.savefig(path_results + "_all_eccentricity.png",dpi=500)
plt.show()

In [ ]:
# Plot the histogram of g ratios
plt.hist(measurements_all['outer_eccentricity'],bins=50, edgecolor='black', color='darkblue', alpha=0.6)

# Customize plot elements
plt.ylabel('Number of cells')
plt.xlabel('Outer Eccentricity')
plt.grid(True, linestyle='--', alpha=0.4, which='both')

# Add additional plot elements
plt.axvline(measurements_all['outer_eccentricity'].mean(), color='red', linestyle='--', label='Mean')
plt.axvline(measurements_all['outer_eccentricity'].median(), color='darkred', linestyle='--', label='Median')
plt.legend()

# Show the plot
plt.tight_layout()
plt.savefig(path_results + "_all_eccentricity.png",dpi=500)
plt.show()

In [ ]:
# Plot the histogram of area
plt.hist(measurements_all['inner_area'],bins=1500, edgecolor='black', color='darkblue', alpha=0.6)
# Customize plot elements
plt.ylabel('Number of cells')
plt.xlabel('Fiber Area [µm²]')
plt.grid(True, linestyle='--', alpha=0.4, which='both')

# Add additional plot elements
plt.axvline((measurements_all['inner_area']).mean(), color='red', linestyle='--', label='Mean')
plt.axvline((measurements_all['inner_area']).median(), color='darkred', linestyle='--', label='Median')
plt.legend()

# Show the plot
plt.tight_layout()
plt.savefig(path_results + "_all_fiber_area.png",dpi=500)
plt.show()

In [ ]:
# Plot the histogram of minor axis length
plt.hist((measurements_all['inner_axis_minor_length']), bins=200, edgecolor='black', color='darkblue', alpha=0.6)
# Customize plot elements
plt.ylabel('Number of cells')
plt.xlabel('Smallest fiber diameter [µm]')
plt.grid(True, linestyle='--', alpha=0.4, which='both')
plt.xticks([0.25,0.5,1,2,3,4,5],[0.25,0.5,1,2,3,4,5])  # Set the x-axis tick positions and labels

# Add additional plot elements
plt.axvline((measurements_all['inner_axis_minor_length']).mean(), color='red', linestyle='--', label='Mean')
plt.axvline((measurements_all['inner_axis_minor_length']).median(), color='darkred', linestyle='--', label='Median')
plt.legend()

# Show the plot
plt.tight_layout()
plt.savefig(path_results + "_all_fiber_diameter.png", dpi=500)
plt.show()
